# Channel Selection by RFE (Recursive Feature Elimination)

**Dataset**: MOABB BNCI2014-001 (Motor Imagery)  
**Channels**: 22 channels  
**Sampling rate**: 250 Hz  
**Subject**: 1

---

## Overview

We use RFE with a linear SVM to select the most important band power features.

## Expected outputs

- Bar chart of features sorted by RFE rank
- Topomap showing selected channels

## Key parameters

| Parameter | Value |
| --- | --- |
| N_SELECT | 10 |
| BANDS | alpha, beta |


## 1. Install dependencies


In [ ]:
!pip install moabb mne scipy numpy plotly scikit-learn


## 2. Load MOABB dataset

MOABB downloads data automatically on first use (~44 MB).


In [ ]:
from moabb.datasets import BNCI2014_001
from moabb.paradigms import MotorImagery
import numpy as np

dataset = BNCI2014_001()
paradigm = MotorImagery(n_classes=2)
X, labels, meta = paradigm.get_data(dataset=dataset, subjects=[1])

mask = (labels == 'left_hand') | (labels == 'right_hand')
X = X[mask]
labels = labels[mask]

print(f'X shape: {X.shape}')
print(f'Labels: {np.unique(labels)}')
print(f'Trials: {len(labels)}')


## 3. Explore the data


In [ ]:
n_trials, n_channels, n_samples = X.shape
print(f'Trials: {n_trials}')
print(f'Channels: {n_channels}')
print(f'Samples per trial: {n_samples}')
print(f'Trial duration: {n_samples/250:.2f} s')


## 4. Apply RFE


In [ ]:
from scipy.signal import welch
from sklearn.svm import SVC
from sklearn.feature_selection import RFE
from sklearn.model_selection import train_test_split

FS = 250
N_SELECT = 10
BANDS = [(8, 13, 'alpha'), (13, 30, 'beta')]

features = np.zeros((n_trials, n_channels * len(BANDS)))
for trial in range(n_trials):
    for ch in range(n_channels):
        freqs, psd = welch(X[trial, ch, :], fs=FS, nperseg=256)
        for b_idx, (fmin, fmax, bname) in enumerate(BANDS):
            mask_f = (freqs >= fmin) & (freqs <= fmax)
            features[trial, ch * len(BANDS) + b_idx] = np.trapezoid(psd[mask_f], freqs[mask_f])

X_train, X_test, y_train, y_test = train_test_split(
    features, labels, test_size=0.2, random_state=42, stratify=labels
)
estimator = SVC(kernel='linear', random_state=42)
selector = RFE(estimator, n_features_to_select=N_SELECT)
selector = selector.fit(X_train, y_train)
selected_mask = selector.support_
print(f'Selected {N_SELECT} features out of {len(selected_mask)}')
print(f"Test accuracy with selected features: {selector.score(X_test, y_test):.4f}")

## 5. Interactive plot

**What to look for:**

- RFE considers interactions between features
- May select different channels than SNR


In [ ]:
import plotly.graph_objects as go

rankings = selector.ranking_
sorted_idx = np.argsort(rankings)
colors = ['green' if selected_mask[i] else 'gray' for i in sorted_idx]
fig = go.Figure(go.Bar(x=[str(i) for i in sorted_idx], y=rankings[sorted_idx], marker_color=colors, name='RFE rank'))
fig.update_layout(height=500, title='RFE Feature Ranking (1=best)', xaxis_title='Feature index', yaxis_title='Rank')
fig.show()


## What did we learn?

- RFE removes features iteratively based on classification importance
- Reveals interactions between features
- More accurate than SNR but slower due to repeated model training
